---
## Section 7 — Transformer model training


In [1]:
# ── Standard library ─────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ── Data manipulation ────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualisation ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# ── NLP — NLTK VADER (lexicon-based sentiment) ───────────────────────────────
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon', quiet=True)   # download lexicon once

# ── Scikit-Learn — feature engineering ──────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Scikit-Learn — model selection ──────────────────────────────────────────
from sklearn.model_selection import train_test_split

# ── Scikit-Learn — model ─────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# ── Scikit-Learn — evaluation ────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# ── Global aesthetics ─────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi'       : 130,
    'axes.titlesize'   : 13,
    'axes.titleweight' : 'bold',
    'axes.labelsize'   : 11,
})

# ── Shared colour palette (consistent with Phase 1) ───────────────────────────
PALETTE = {
    'Negative' : '#E05C5C',
    'Neutral'  : '#F5A623',
    'Positive' : '#4CAF82',
}
LABEL_ORDER = ['Negative', 'Neutral', 'Positive']

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('  Environment ready.')

  Environment ready.


---
## Section 1 — Data Loading


In [2]:
# ── Load Phase 1 output ───────────────────────────────────────────────────────
CSV_PATH = 'movie_reviews_phase1_cleaned.csv'   
df = pd.read_csv(CSV_PATH)

print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')

Loaded: 87,906 rows × 12 columns
Columns: ['review_id', 'reviewer', 'movie', 'rating', 'review_summary', 'review_date', 'spoiler_tag', 'review_detail', 'helpful', 'sentiment_label', 'cleaned_text', 'review_length']


In [3]:
import torch
import numpy as np
import evaluate
import accelerate
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding
)

In [18]:
# 1. Verify GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: No GPU detected! BERT training will be extremely slow. Switch your runtime to GPU.")

Using device: cuda


In [5]:
relevant_df= df[['cleaned_text', 'sentiment_label']].copy()

In [6]:
# 2. Extract arrays and map string labels ('negative', 'neutral', 'positive') to integers
label_mapping = {'Negative': 0, 'Neutral': 1, 'Positive': 2}

texts = relevant_df['cleaned_text'].astype(str).tolist()
labels = [label_mapping[lbl] for lbl in relevant_df['sentiment_label']]

In [7]:
# Extract a small balanced 15,000 row subset to guarantee lightning speed
_, X_small, _, y_small = train_test_split(
    texts, labels, test_size=15000, stratify=labels, random_state=42
)

# Split subset into 80% Train / 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    X_small, y_small, test_size=0.20, stratify=y_small, random_state=42
)

In [8]:
# 4. Construct light-weight Hugging Face Datasets from the split columns
train_dataset = Dataset.from_dict({"text": X_train, "label": y_train})
test_dataset = Dataset.from_dict({"text": X_test, "label": y_test})

In [9]:
# 5. Tokenize using 'bert-base-uncased'
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [10]:
def tokenize_function(examples):
    # max_length=256 significantly improves training speeds over 512
    return tokenizer(examples["text"], truncation=True, max_length=256)

print("Tokenizing isolated columns...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Tokenizing isolated columns...


Map: 100%|██████████| 3000/3000 [00:00<00:00, 5118.22 examples/s]


In [11]:
# Pads arrays dynamically to the longest element in a batch (saves matrix memory)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [19]:
# 6. Initialize BERT Model (Targeting 3 classes)
device = ""
if torch.cuda.is_available() :
    device= "cuda"
    print("cuda is available")
else:
    "cpu"
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=3
).to(device)

cuda is available


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2229.56it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [20]:
# 7. Setup standard Macro F1 evaluation metric
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels, average="macro")

In [21]:
# 8. Low-Intensity Training Arguments
training_args = TrainingArguments(
    output_dir="./bert_fast_model",
    learning_rate=3e-5,                  
    per_device_train_batch_size=32,     # Maximizes parallel computing capability
    per_device_eval_batch_size=32,
    num_train_epochs=1,                 # Only 1 pass over the small subset
    weight_decay=0.01,
    eval_strategy="no",                 
    save_strategy="no",                 
    fp16=torch.cuda.is_available(),     # Activates dynamic half-precision floating point math
    logging_steps=50,
    report_to="none",
    ignore_data_skip=True
)

In [22]:
# 9. Initialize and Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [25]:
# 9. Clear explicit memory and force initialize
import gc
torch.cuda.empty_cache() if torch.cuda.is_available() else None
gc.collect()

6207

In [24]:
print(f"Starting BERT training on device: {device}")
trainer.train()

Starting BERT training on device: cuda


Step,Training Loss


KeyboardInterrupt: 